# CIDS · 12 — S3f: memory jury as a saved model + final masks for every case

Turns the S3e result (13-clue memory jury, test Dice **0.7495** from tables) into a **real model and real masks**:
- runs the automatic pipeline once (SegResNet-100 → candidates → SAM-S2b → 13 clues), fits the memory jury on train, chooses the cut-off on **val**, applies it once to **test**;
- writes the **final mask of every case** — val/test from the deployable jury, **train from an out-of-fold jury** (5 folds by patient, never saw that patient) → input of the boundary refiner (notebooks 13 / 14);
- **full metrics** on val/test (Dice, HD95, NSD, missed) for memory jury vs S3d jury vs SegResNet rel20, Wilcoxon + bootstrap CI;
- checks that mask Dice equals the table Dice (parity), saves `memory_jury.pkl`.

**Settings:** GPU T4 x2 · Internet ON. **Inputs:** cache · `cids-checkpoints`. **Time:** ~2.5–3 h. Resumable per split (same manifest).
**After the run:** this notebook's output (`/kaggle/working/s3f_jury`, ~3 GB: `masks/`, `cases/`) is the input of the refiner notebooks.

In [ ]:
REPO_URL            = 'https://github.com/Vemuri-BK/cids.git'
GIT_BRANCH          = 'main'
EXPECTED_GIT_COMMIT = ''      # paste the full commit id you pushed

CACHE_ROOT = '/kaggle/input/datasets/kumarvemuri1/mama-mia-preprocessed-cache-2ch'
CKPT_DIR   = '/kaggle/input/datasets/kumarvemuri1/cids-checkpoints'
SEG_CKPT   = f'{CKPT_DIR}/segresnet_img_e100_best.pt'
SAM_CKPT   = f'{CKPT_DIR}/sam_s2b_best.pt'
CODE_DIR   = '/kaggle/working/cids'
OUT        = '/kaggle/working/s3f_jury'
K          = 10                # nearest memories
PATHS = [CACHE_ROOT, SEG_CKPT, SAM_CKPT]

In [ ]:
import os, subprocess
if os.path.isdir(f'{CODE_DIR}/.git'):
    subprocess.run(['git', '-C', CODE_DIR, 'fetch', 'origin', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', GIT_BRANCH], check=True)
    subprocess.run(['git', '-C', CODE_DIR, 'pull', 'origin', GIT_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '-b', GIT_BRANCH, REPO_URL, CODE_DIR], check=True)
if EXPECTED_GIT_COMMIT:
    subprocess.run(['git', '-C', CODE_DIR, 'checkout', EXPECTED_GIT_COMMIT], check=True)
head = subprocess.check_output(['git', '-C', CODE_DIR, 'rev-parse', 'HEAD'], text=True).strip()
assert not EXPECTED_GIT_COMMIT or head == EXPECTED_GIT_COMMIT, (head, EXPECTED_GIT_COMMIT)
print('code at commit', head)
%cd {CODE_DIR}

def sh(cmd):
    """Run a shell step, stream its output, and STOP the notebook if it fails (plain !cmd does not)."""
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    if p.wait():
        raise RuntimeError(f'step failed (exit {p.returncode}): {cmd}')

In [ ]:
sh(f'pip install -q monai medim einops appdirs')
sh(f'pytest -q tests')

In [ ]:
import torch
from pathlib import Path
assert torch.cuda.is_available(), 'No GPU! Settings -> Accelerator -> GPU T4 x2, then restart.'
for q in PATHS:
    print('OK ' if Path(q).exists() else '!! MISSING', q)
assert all(Path(q).exists() for q in PATHS)

## Smoke test (~6 min): 10 cases per split
Check: the val cut-off line, two VAL / TEST tables (memory_jury, s3d_jury11, seg_rel20), `parity … 0.0`. Numbers on 10 cases mean nothing.

In [ ]:
sh(f'python scripts/s3f_jury_masks.py --cache_root {CACHE_ROOT} --seg_ckpt {SEG_CKPT} --sam_ckpt {SAM_CKPT} --out /kaggle/working/s3f_smoke --limit 10 --k 3')
sh(f'rm -rf /kaggle/working/s3f_smoke')

## Full run (~2.5–3 h)
**Read:** memory_jury test Dice ≈ **0.7495** (same as notebook 11), s3d_jury11 ≈ **0.735**; HD95 / NSD for both; `parity` must be 0.0.
`train out-of-fold mask Dice` = quality of the refiner's training masks (should be close to val/test, not much higher).

In [ ]:
sh(f'python scripts/s3f_jury_masks.py --cache_root {CACHE_ROOT} --seg_ckpt {SEG_CKPT} --sam_ckpt {SAM_CKPT} --out {OUT} --k {K} --zip_outputs')

In [ ]:
import json
s = json.load(open(f'{OUT}/summary.json'))
print(json.dumps({k: s[k] for k in ('threshold', 'val_dice', 'parity_max_abs_diff', 'train_oof_table_dice', 'auc_train_oof', 'auc_val', 'auc_test', 'cases', 'minutes')}, indent=2))
for split in ('val', 'test'):
    print(split, json.dumps(s[split]['metrics'], indent=1))
    print(split, 'memory jury vs:', json.dumps(s[split]['memory_jury_vs'], indent=1))
sh(f'du -sh {OUT} && ls -la {OUT}')